In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

In [2]:
product_df = pd.read_csv("cleaned_product.csv")

print("Dataset Shape:", product_df.shape)

Dataset Shape: (105542, 23)


In [3]:
selected_features = [
    "product_type_name",
    "section_name",
    "detail_desc",
    "graphical_appearance_name"
]


In [4]:
for col in selected_features:
    product_df[col] = product_df[col].fillna("")

In [5]:
product_df["combined_text"] = (
    product_df["product_type_name"] + " " +
    product_df["section_name"] + " " +
    product_df["graphical_appearance_name"] + " " +
    product_df["detail_desc"]
)

In [6]:
unique_products = product_df.drop_duplicates(subset=["prod_name"])

print("Original rows:", len(product_df))
print("Unique products:", len(unique_products))

Original rows: 105542
Unique products: 45875


In [7]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1,2)
)

tfidf_matrix = tfidf.fit_transform(unique_products["combined_text"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (45875, 20000)


In [8]:
nn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn_model.fit(tfidf_matrix)

print("Nearest Neighbor Model Trained")

Nearest Neighbor Model Trained


In [9]:
if "purchase_count" in product_df.columns:
    scaler = MinMaxScaler()
    product_df["popularity_score"] = scaler.fit_transform(
        product_df[["purchase_count"]]
    )
else:
    product_df["popularity_score"] = 0.5  # neutral value

In [10]:
def recommend_products(product_name, top_n=5):

    if product_name not in product_df["prod_name"].values:
        return "Product not found"

    idx = product_df[
        product_df["prod_name"] == product_name
    ].index[0]

    distances, indices = nn_model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=top_n + 10  # get extra for ranking
    )

    similarity_scores = 1 - distances.flatten()

    candidate_indices = indices.flatten()

    # Remove selected product
    filtered = [
        (i, sim) for i, sim in zip(candidate_indices, similarity_scores)
        if i != idx
    ]

    # Convert to DataFrame
    rec_df = pd.DataFrame(filtered, columns=["index", "similarity"])

    rec_df["popularity"] = product_df.loc[
        rec_df["index"], "popularity_score"
    ].values

    # Hybrid scoring
    rec_df["final_score"] = (
        0.7 * rec_df["similarity"] +
        0.3 * rec_df["popularity"]
    )

    # Sort by final score
    rec_df = rec_df.sort_values(
        by="final_score",
        ascending=False
    )

    top_indices = rec_df.head(top_n)["index"]

    return product_df.loc[
        top_indices,
        [
            "prod_name",
            "product_type_name",
            "section_name",
            "colour_group_name"
        ]
    ]


In [11]:
sample_product = product_df["prod_name"].iloc[0]
print("Sample Product:", sample_product)

recommendations = recommend_products(sample_product, top_n=5)
print(recommendations)

Sample Product: Strap top
              prod_name         product_type_name  \
3750   Cindy N-slip (K)                Night gown   
34816          3P SS PJ  Pyjama jumpsuit/playsuit   
19822       Bella dress                     Dress   
29894    WENDY LEGGINGS           Leggings/Tights   
25940               Amy                    Blouse   

                         section_name colour_group_name  
3750   Womens Nightwear, Socks & Tigh        Light Pink  
34816   Baby Essentials & Complements              Pink  
19822                       Kids Girl        Light Pink  
29894                      Young Girl         Dark Grey  
25940                            Mama        Light Pink  


In [12]:
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(nn_model, "nn_model.pkl")

product_df.to_csv("recommendation_products.csv", index=False)

print("Recommendation System Saved Successfully")

Recommendation System Saved Successfully
